In [ ]:
# ============================================================
# Individual 3-class classification for 7 quality indicators
#
# For each quality indicator:
#   1. Use the SAME fixed 113 training samples and 36 validation samples
#      as the protein regression model.
#   2. Use only the training 113 samples to fit KMeans3 on the target indicator.
#   3. Assign validation 36 samples to the nearest KMeans3 class.
#   4. Train HSI classification models using patch-level spectra.
#   5. Evaluate:
#        - Original model performance: GroupKFold on training 113
#        - Fixed validation performance: same 36 validation samples
#
# Important:
#   The 7 quality indicators are NOT model inputs.
#   They are only used to define class labels.
#
# Spectral input:
#   patch-level HSI spectra
#
# Classes:
#   Class I   = Low target value
#   Class II  = Medium target value
#   Class III = High target value
# ============================================================

import os
import re
import json
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
)

import joblib


# ============================================================
# 0. Settings
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K_CLASSES = 3


def find_project_root(start_path=None):
    """
    Locate the repository root by searching upward for the
    'Pea samples-new' input-data directory.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "Pea samples-new").is_dir():
            return candidate

    raise FileNotFoundError(
        "Cannot locate the project root. The repository must contain "
        "a folder named 'Pea samples-new', and Jupyter must be started "
        "from the repository or one of its subfolders."
    )


PROJECT_DIR = find_project_root()
DATA_BASE = PROJECT_DIR / "Pea samples-new"
MODEL_DIR = DATA_BASE / "Regression model"

# Existing raw/input files only
A_PATCH_CSV = DATA_BASE / "pea_patch_dataset.csv"
B_PATCH_CSV = MODEL_DIR / "flour_external_pea_patch_dataset.csv"
B_QUALITY_XLSX = MODEL_DIR / "pea quality-2025.xlsx"

OUT_ROOT = PROJECT_DIR / "Pea_TwoSource_Mixed113_IndividualIndicator_KMeans3_Classification"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_COLS = [
    "Protein content",
    "Moisture content",
    "Water uptaking capacity",
    "Water binding capacity",
    "Oil binding capacity",
    "Protein solubility",
    "Peak viscosity (RVA)",
]

CLASS_ID_TO_NAME = {
    0: "Class I",
    1: "Class II",
    2: "Class III",
}

CLASS_ID_TO_LEVEL = {
    0: "Low",
    1: "Medium",
    2: "High",
}

QUALITY_COLS_ALL = TARGET_COLS.copy()

MODEL_NAMES_TO_RUN = [
    "LogReg",
    "SVM_RBF",
    "RandomForest",
    "ExtraTrees",
    "SoftVoting",
]

# If a target has missing values in the fixed protein split:
# False = stop and report missing samples
# True  = keep the same split IDs but drop samples with missing target values
DROP_MISSING_TARGET_IN_SAME_SPLIT = False


# ============================================================
# 1. Figure style
# ============================================================

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["xtick.labelsize"] = 20
plt.rcParams["ytick.labelsize"] = 20
plt.rcParams["legend.fontsize"] = 20


# ============================================================
# 2. Basic helper functions
# ============================================================

def find_existing_path(path_candidates):
    for p in path_candidates:
        if Path(p).exists():
            return Path(p)

    raise FileNotFoundError(
        "None of the candidate files were found:\n" +
        "\n".join([str(p) for p in path_candidates])
    )


def clean_sample_id(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s.endswith(".0"):
        s = s[:-2]

    return s


def safe_name(s):
    s = str(s)
    s = s.replace("%", "pct")
    s = re.sub(r"[^\w\-]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")


def detect_sample_id_column(df):
    candidates = [
        "sample_id",
        "Sample ID",
        "Sample_ID",
        "Sample",
        "sample",
        "ID",
        "id",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "Cannot detect sample id column. Available columns:\n"
        f"{list(df.columns)}"
    )


def detect_wavelength_columns(df):
    """
    Detect spectral wavelength columns.

    Supports:
        997.71
        997.71nm
        997.71 nm
        X997.71
        wavelength_997.71
        band_997.71
    """

    wave_cols = []
    wave_values = []

    for c in df.columns:
        c_str = str(c).strip()

        try:
            w = float(c_str)
            if 900 <= w <= 2600:
                wave_cols.append(c)
                wave_values.append(w)
                continue
        except Exception:
            pass

        nums = re.findall(r"\d+\.\d+|\d+", c_str)

        if len(nums) > 0:
            try:
                w = float(nums[0])

                if 900 <= w <= 2600:
                    wave_cols.append(c)
                    wave_values.append(w)

            except Exception:
                pass

    if len(wave_cols) == 0:
        raise ValueError(
            "No wavelength columns detected.\n\n"
            "First 30 columns:\n"
            f"{list(df.columns[:30])}\n\n"
            "Last 30 columns:\n"
            f"{list(df.columns[-30:])}"
        )

    order = np.argsort(wave_values)

    wave_cols = [wave_cols[i] for i in order]
    wave_values = np.array([wave_values[i] for i in order], dtype=float)

    print(f"Detected {len(wave_cols)} wavelength columns.")
    print("First 5 wavelengths:", wave_values[:5])
    print("Last 5 wavelengths:", wave_values[-5:])

    return wave_cols, wave_values


def normalize_quality_column_names(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    rename_map = {}

    for c in df.columns:
        c_low = c.lower().strip()

        if c_low in [
            "protein",
            "protein content",
            "protein_content",
        ]:
            rename_map[c] = "Protein content"

        elif c_low in [
            "moisture",
            "moisture content",
            "moisture_content",
        ]:
            rename_map[c] = "Moisture content"

        elif c_low in [
            "water uptaking capacity",
            "water uptake capacity",
            "wuc",
            "water_uptaking_capacity",
            "water_uptake_capacity",
        ]:
            rename_map[c] = "Water uptaking capacity"

        elif c_low in [
            "water binding capacity",
            "wbc",
            "water_binding_capacity",
        ]:
            rename_map[c] = "Water binding capacity"

        elif c_low in [
            "oil binding capacity",
            "obc",
            "oil_binding_capacity",
        ]:
            rename_map[c] = "Oil binding capacity"

        elif c_low in [
            "protein solubility",
            "solubility",
            "protein_solubility",
        ]:
            rename_map[c] = "Protein solubility"

        elif c_low in [
            "peak viscosity",
            "peak viscosity (rva)",
            "rva",
            "peak_viscosity",
            "peak_viscosity_rva",
        ]:
            rename_map[c] = "Peak viscosity (RVA)"

    df = df.rename(columns=rename_map)

    return df


def add_b_treatment_from_sample_id(df):
    df = df.copy()

    def map_treatment(sid):
        try:
            sid_int = int(float(str(sid)))
        except Exception:
            return "Unknown"

        if 1 <= sid_int <= 12:
            return "Control"
        elif 13 <= sid_int <= 24:
            return "150% N"
        elif 25 <= sid_int <= 36:
            return "50% water"
        else:
            return "Unknown"

    df["treatment"] = df["sample_id"].apply(map_treatment)

    return df


# ============================================================
# 3. Data loading functions
# ============================================================

def prepare_patch_table(patch_csv, source_name):
    patch_df = pd.read_csv(patch_csv)
    patch_df.columns = [str(c).strip() for c in patch_df.columns]

    sample_col = detect_sample_id_column(patch_df)
    patch_df = patch_df.rename(columns={sample_col: "sample_id"})
    patch_df["sample_id"] = patch_df["sample_id"].apply(clean_sample_id)

    patch_df["source"] = source_name

    patch_df["global_sample_id"] = np.where(
        source_name == "A_original",
        "A_" + patch_df["sample_id"].astype(str),
        "B_" + patch_df["sample_id"].astype(str),
    )

    if source_name == "B_second":
        patch_df = add_b_treatment_from_sample_id(patch_df)
    else:
        patch_df["treatment"] = "Original"

    wave_cols, wavelengths = detect_wavelength_columns(patch_df)

    meta_cols = ["global_sample_id", "source", "sample_id", "treatment"]

    keep_cols = meta_cols + wave_cols

    patch_df = patch_df[keep_cols].copy()

    return patch_df, wave_cols, wavelengths


def build_quality_all_from_patch_and_excel(a_patch_full_csv, b_quality_xlsx):
    """
    Build combined A+B quality table directly in memory.
    A quality is read from the A patch CSV because the original workflow
    stores the 2024 quality columns there.
    B quality is read from the 2025 quality Excel file.
    """

    a_quality = pd.read_csv(a_patch_full_csv)
    a_quality.columns = [str(c).strip() for c in a_quality.columns]
    a_quality = normalize_quality_column_names(a_quality)

    sample_col = detect_sample_id_column(a_quality)
    a_quality = a_quality.rename(columns={sample_col: "sample_id"})
    a_quality["sample_id"] = a_quality["sample_id"].apply(clean_sample_id)

    a_quality["source"] = "A_original"
    a_quality["global_sample_id"] = "A_" + a_quality["sample_id"].astype(str)
    a_quality["treatment"] = "Original"

    missing_a_quality = [c for c in QUALITY_COLS_ALL if c not in a_quality.columns]

    if len(missing_a_quality) > 0:
        raise ValueError(
            "A patch CSV does not contain all required quality columns.\n"
            f"Missing A quality columns: {missing_a_quality}"
        )

    a_quality = (
        a_quality[
            ["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS_ALL
        ]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    b_quality = pd.read_excel(b_quality_xlsx)
    b_quality = normalize_quality_column_names(b_quality)

    sample_col = detect_sample_id_column(b_quality)
    b_quality = b_quality.rename(columns={sample_col: "sample_id"})
    b_quality["sample_id"] = b_quality["sample_id"].apply(clean_sample_id)

    b_quality["source"] = "B_second"
    b_quality["global_sample_id"] = "B_" + b_quality["sample_id"].astype(str)

    if "treatment" not in b_quality.columns:
        b_quality = add_b_treatment_from_sample_id(b_quality)

    missing_b_quality = [c for c in QUALITY_COLS_ALL if c not in b_quality.columns]

    if len(missing_b_quality) > 0:
        raise ValueError(
            f"B quality Excel is missing quality columns: {missing_b_quality}\n"
            f"Available columns: {list(b_quality.columns)}"
        )

    b_quality = (
        b_quality[
            ["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS_ALL
        ]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    quality_all = pd.concat([a_quality, b_quality], axis=0, ignore_index=True)

    return quality_all


def load_quality_all_from_original_inputs(a_patch_csv, b_quality_xlsx):
    """
    Build the 149-sample quality table directly in memory.

    No combined quality CSV is required or written.
    """
    quality_all = build_quality_all_from_patch_and_excel(
        a_patch_full_csv=a_patch_csv,
        b_quality_xlsx=b_quality_xlsx,
    )

    quality_all["source"] = quality_all["source"].astype(str)
    quality_all["global_sample_id"] = quality_all["global_sample_id"].astype(str)
    quality_all["sample_id"] = quality_all["sample_id"].apply(clean_sample_id)

    for c in QUALITY_COLS_ALL:
        quality_all[c] = pd.to_numeric(quality_all[c], errors="coerce")

    quality_all = (
        quality_all[
            ["global_sample_id", "source", "sample_id", "treatment"] + QUALITY_COLS_ALL
        ]
        .drop_duplicates(subset=["global_sample_id"])
        .reset_index(drop=True)
    )

    source_counts = quality_all["source"].value_counts().to_dict()

    if source_counts.get("A_original", 0) != 113:
        raise ValueError(
            "Expected 113 Source A samples, but found "
            f"{source_counts.get('A_original', 0)}."
        )

    if source_counts.get("B_second", 0) != 36:
        raise ValueError(
            "Expected 36 Source B samples, but found "
            f"{source_counts.get('B_second', 0)}."
        )

    if quality_all["global_sample_id"].nunique() != 149:
        raise ValueError(
            "Expected 149 unique samples after combining the two years, "
            f"but found {quality_all['global_sample_id'].nunique()}."
        )

    return quality_all


# ============================================================
# 4. Fixed 113/36 split
# ============================================================

def create_fixed_split_repeat0(quality_all):
    """
    Recreate the exact repeat-0 split used by the protein regression code.

    Training:
        89 Source A samples + 24 Source B samples = 113

    Independent validation:
        24 Source A samples + 12 Source B samples = 36

    The 12 Source B validation samples contain:
        4 Control + 4 150% N + 4 50% water

    No split-ID files are required or written.
    """
    rng = np.random.default_rng(RANDOM_STATE + 0)

    quality_a = quality_all[quality_all["source"] == "A_original"].copy()
    quality_b = quality_all[quality_all["source"] == "B_second"].copy()

    a_ids = np.array(sorted(
        quality_a["global_sample_id"].astype(str).unique()
    ))

    if len(a_ids) != 113:
        raise ValueError(
            f"Expected 113 Source A samples, but found {len(a_ids)}."
        )

    a_val_ids = rng.choice(a_ids, size=24, replace=False)
    a_val_set = set(a_val_ids.tolist())
    a_train_ids = np.array([sid for sid in a_ids if sid not in a_val_set])

    b_val_ids = []
    b_train_ids = []

    for treatment in ["Control", "150% N", "50% water"]:
        ids_t = np.array(sorted(
            quality_b.loc[
                quality_b["treatment"] == treatment,
                "global_sample_id",
            ].astype(str).unique()
        ))

        if len(ids_t) != 12:
            raise ValueError(
                f"Expected 12 Source B samples for treatment '{treatment}', "
                f"but found {len(ids_t)}."
            )

        val_t = rng.choice(ids_t, size=4, replace=False)
        val_t_set = set(val_t.tolist())
        train_t = np.array([sid for sid in ids_t if sid not in val_t_set])

        b_val_ids.extend(val_t.tolist())
        b_train_ids.extend(train_t.tolist())

    train_ids = sorted(a_train_ids.tolist() + b_train_ids)
    val_ids = sorted(a_val_ids.tolist() + b_val_ids)

    if len(train_ids) != 113:
        raise ValueError(
            f"Training split should contain 113 samples, but found {len(train_ids)}."
        )

    if len(val_ids) != 36:
        raise ValueError(
            f"Validation split should contain 36 samples, but found {len(val_ids)}."
        )

    overlap = set(train_ids).intersection(val_ids)

    if overlap:
        raise ValueError(
            f"Training and validation samples overlap: {sorted(overlap)}"
        )

    print("\nFixed split recreated in memory.")
    print("Training samples:", len(train_ids))
    print("Validation samples:", len(val_ids))

    print("\nTraining source counts:")
    print(
        quality_all.loc[
            quality_all["global_sample_id"].isin(train_ids),
            "source",
        ].value_counts()
    )

    print("\nValidation source counts:")
    print(
        quality_all.loc[
            quality_all["global_sample_id"].isin(val_ids),
            "source",
        ].value_counts()
    )

    print("\nValidation Source B treatment counts:")
    print(
        quality_all.loc[
            quality_all["global_sample_id"].isin(val_ids)
            & (quality_all["source"] == "B_second"),
            "treatment",
        ].value_counts()
    )

    return train_ids, val_ids


# ============================================================
# 5. Spectral preprocessing and model functions
# ============================================================

class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        row_mean = np.nanmean(X, axis=1, keepdims=True)
        row_std = np.nanstd(X, axis=1, keepdims=True)

        row_std[row_std == 0] = 1.0

        return (X - row_mean) / row_std


def make_classifier(model_name):
    if model_name == "LogReg":
        model = LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )

    elif model_name == "SVM_RBF":
        model = SVC(
            C=10.0,
            gamma="scale",
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )

    elif model_name == "RandomForest":
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif model_name == "ExtraTrees":
        model = ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif model_name == "SoftVoting":
        logreg = LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )

        svm = SVC(
            C=10.0,
            gamma="scale",
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )

        et = ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        model = VotingClassifier(
            estimators=[
                ("logreg", logreg),
                ("svm", svm),
                ("extratrees", et),
            ],
            voting="soft",
            weights=[1, 1, 1],
            n_jobs=-1,
        )

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    pipeline = Pipeline(
        steps=[
            ("snv", SNVTransformer()),
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )

    return pipeline


# ============================================================
# 6. Label creation functions
# ============================================================

def create_kmeans3_labels_for_target(
    quality_all,
    train_ids,
    val_ids,
    target_col,
):
    q = quality_all.copy()

    q[target_col] = pd.to_numeric(q[target_col], errors="coerce")

    q_train = q[
        q["global_sample_id"].astype(str).isin(train_ids)
    ].copy()

    q_val = q[
        q["global_sample_id"].astype(str).isin(val_ids)
    ].copy()

    missing_train = q_train.loc[
        q_train[target_col].isna(),
        "global_sample_id"
    ].astype(str).tolist()

    missing_val = q_val.loc[
        q_val[target_col].isna(),
        "global_sample_id"
    ].astype(str).tolist()

    if len(missing_train) > 0 or len(missing_val) > 0:
        print(f"\nMissing values detected for target: {target_col}")
        print("Training missing IDs:", missing_train)
        print("Validation missing IDs:", missing_val)

        if not DROP_MISSING_TARGET_IN_SAME_SPLIT:
            raise ValueError(
                f"Missing target values exist for {target_col}. "
                "Set DROP_MISSING_TARGET_IN_SAME_SPLIT=True if you want "
                "to keep the same protein split IDs but drop samples with "
                "missing target values for this target."
            )

        q_train = q_train.dropna(subset=[target_col]).copy()
        q_val = q_val.dropna(subset=[target_col]).copy()

    scaler_y = StandardScaler()

    y_train_std = scaler_y.fit_transform(
        q_train[[target_col]].values.astype(float)
    )

    kmeans_y = KMeans(
        n_clusters=K_CLASSES,
        random_state=RANDOM_STATE,
        n_init=50,
    )

    raw_train_label = kmeans_y.fit_predict(y_train_std)

    raw_centers_std = kmeans_y.cluster_centers_.reshape(-1, 1)

    raw_centers_original = scaler_y.inverse_transform(
        raw_centers_std
    ).ravel()

    center_order = np.argsort(raw_centers_original)

    raw_to_ordered = {
        int(raw_label): int(ordered_label)
        for ordered_label, raw_label in enumerate(center_order)
    }

    q_train["class_id"] = [
        raw_to_ordered[int(x)]
        for x in raw_train_label
    ]

    y_val_std = scaler_y.transform(
        q_val[[target_col]].values.astype(float)
    )

    raw_val_label = kmeans_y.predict(y_val_std)

    q_val["class_id"] = [
        raw_to_ordered[int(x)]
        for x in raw_val_label
    ]

    q_train["split"] = "training"
    q_val["split"] = "validation"

    label_df = pd.concat(
        [q_train, q_val],
        axis=0,
        ignore_index=True,
    )

    label_df["class_id"] = label_df["class_id"].astype(int)

    label_df["class_name"] = label_df["class_id"].map(
        CLASS_ID_TO_NAME
    )

    label_df["class_level"] = label_df["class_id"].map(
        CLASS_ID_TO_LEVEL
    )

    ordered_centers = np.sort(raw_centers_original)

    centers_df = pd.DataFrame(
        {
            "class_id": np.arange(K_CLASSES, dtype=int),
            "class_name": [
                CLASS_ID_TO_NAME[i]
                for i in range(K_CLASSES)
            ],
            "class_level": [
                CLASS_ID_TO_LEVEL[i]
                for i in range(K_CLASSES)
            ],
            "center_value_original_scale": ordered_centers,
        }
    )

    train_count_df = (
        q_train["class_id"]
        .value_counts()
        .reindex(range(K_CLASSES), fill_value=0)
        .sort_index()
        .rename("training_n")
        .reset_index()
        .rename(columns={"index": "class_id"})
    )

    val_count_df = (
        q_val["class_id"]
        .value_counts()
        .reindex(range(K_CLASSES), fill_value=0)
        .sort_index()
        .rename("validation_n")
        .reset_index()
        .rename(columns={"index": "class_id"})
    )

    class_count_df = train_count_df.merge(
        val_count_df,
        on="class_id",
        how="outer",
    )

    class_count_df["class_name"] = class_count_df["class_id"].map(
        CLASS_ID_TO_NAME
    )

    class_count_df["class_level"] = class_count_df["class_id"].map(
        CLASS_ID_TO_LEVEL
    )

    class_count_df = class_count_df[
        [
            "class_id",
            "class_name",
            "class_level",
            "training_n",
            "validation_n",
        ]
    ]

    keep_cols = [
        "global_sample_id",
        "source",
        "sample_id",
        "treatment",
        target_col,
        "split",
        "class_id",
        "class_name",
        "class_level",
    ]

    label_df = label_df[keep_cols].copy()

    return (
        label_df,
        centers_df,
        class_count_df,
        scaler_y,
        kmeans_y,
        raw_to_ordered,
    )


def prepare_patch_for_classification(
    patch_all,
    label_df,
):
    label_keep = label_df[
        [
            "global_sample_id",
            "class_id",
            "class_name",
            "class_level",
        ]
    ].copy()

    patch_labeled = patch_all.merge(
        label_keep,
        on="global_sample_id",
        how="inner",
        validate="many_to_one",
    )

    return patch_labeled


# ============================================================
# 7. Prediction and metric functions
# ============================================================

def aggregate_patch_probabilities_to_sample(
    patch_df,
    patch_probabilities,
    class_ids,
):
    prob_cols = [
        f"prob_class_{class_id}"
        for class_id in class_ids
    ]

    prob_df = pd.DataFrame(
        patch_probabilities,
        columns=prob_cols,
        index=patch_df.index,
    )

    meta_df = patch_df[
        [
            "global_sample_id",
            "source",
            "sample_id",
            "treatment",
            "class_id",
            "class_name",
            "class_level",
        ]
    ].copy()

    temp = pd.concat(
        [meta_df, prob_df],
        axis=1,
    )

    agg_dict = {
        "source": "first",
        "sample_id": "first",
        "treatment": "first",
        "class_id": "first",
        "class_name": "first",
        "class_level": "first",
    }

    for c in prob_cols:
        agg_dict[c] = "mean"

    sample_pred = (
        temp
        .groupby("global_sample_id", as_index=False)
        .agg(agg_dict)
    )

    prob_matrix = sample_pred[prob_cols].values

    pred_position = np.argmax(prob_matrix, axis=1)

    sample_pred["predicted_class_id"] = [
        class_ids[i]
        for i in pred_position
    ]

    sample_pred["predicted_class_name"] = sample_pred[
        "predicted_class_id"
    ].map(CLASS_ID_TO_NAME)

    sample_pred["predicted_class_level"] = sample_pred[
        "predicted_class_id"
    ].map(CLASS_ID_TO_LEVEL)

    return sample_pred


def calculate_classification_metrics(
    sample_pred,
    class_ids,
):
    y_true = sample_pred["class_id"].astype(int).values
    y_pred = sample_pred["predicted_class_id"].astype(int).values

    metrics = {
        "n_samples": len(sample_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=class_ids,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=class_ids,
            average="weighted",
            zero_division=0,
        ),
    }

    return metrics


def classification_report_df(
    sample_pred,
    class_ids,
):
    y_true = sample_pred["class_id"].astype(int).values
    y_pred = sample_pred["predicted_class_id"].astype(int).values

    report = classification_report(
        y_true,
        y_pred,
        labels=class_ids,
        target_names=[
            CLASS_ID_TO_NAME[c]
            for c in class_ids
        ],
        zero_division=0,
        output_dict=True,
    )

    return pd.DataFrame(report).T


def plot_confusion_matrix(
    sample_pred,
    class_ids,
    output_path,
):
    y_true = sample_pred["class_id"].astype(int).values
    y_pred = sample_pred["predicted_class_id"].astype(int).values

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_ids,
    )

    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(cm)

    ax.set_xticks(np.arange(len(class_ids)))
    ax.set_yticks(np.arange(len(class_ids)))

    ax.set_xticklabels(
        [CLASS_ID_TO_NAME[c] for c in class_ids]
    )

    ax.set_yticklabels(
        [CLASS_ID_TO_NAME[c] for c in class_ids]
    )

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")

    threshold = cm.max() / 2 if cm.max() > 0 else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm[i, j] > threshold else "black"

            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color=text_color,
                fontsize=20,
            )

    fig.colorbar(image, ax=ax)

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.show()
    plt.close(fig)

    return cm


def groupkfold_oof_classification(
    patch_train,
    wave_cols,
    model_name,
    class_ids,
    n_splits=5,
):
    groups = patch_train["global_sample_id"].astype(str).values

    unique_groups = np.unique(groups)

    n_splits = min(n_splits, len(unique_groups))

    if n_splits < 2:
        raise ValueError("At least 2 unique samples are required for GroupKFold.")

    gkf = GroupKFold(n_splits=n_splits)

    fold_sample_predictions = []

    X_all = patch_train[wave_cols].values.astype(float)
    y_all = patch_train["class_id"].astype(int).values

    for fold_id, (train_idx, test_idx) in enumerate(
        gkf.split(X_all, y_all, groups)
    ):
        fold_train = patch_train.iloc[train_idx].copy()
        fold_test = patch_train.iloc[test_idx].copy()

        model = make_classifier(model_name)

        X_train = fold_train[wave_cols].values.astype(float)
        y_train = fold_train["class_id"].astype(int).values

        X_test = fold_test[wave_cols].values.astype(float)

        model.fit(X_train, y_train)

        patch_proba_raw = model.predict_proba(X_test)

        model_classes = model.named_steps["model"].classes_.astype(int)

        aligned_proba = np.zeros(
            (len(fold_test), len(class_ids)),
            dtype=float,
        )

        for j, cls in enumerate(model_classes):
            class_position = class_ids.index(int(cls))
            aligned_proba[:, class_position] = patch_proba_raw[:, j]

        sample_pred_fold = aggregate_patch_probabilities_to_sample(
            patch_df=fold_test,
            patch_probabilities=aligned_proba,
            class_ids=class_ids,
        )

        sample_pred_fold["fold"] = fold_id

        fold_sample_predictions.append(sample_pred_fold)

    oof_sample_pred = pd.concat(
        fold_sample_predictions,
        axis=0,
        ignore_index=True,
    )

    metrics = calculate_classification_metrics(
        sample_pred=oof_sample_pred,
        class_ids=class_ids,
    )

    metrics["model"] = model_name
    metrics["scenario"] = "internal_GroupKFold"
    metrics["n_splits"] = n_splits

    return metrics, oof_sample_pred


def fit_final_and_validate_classifier(
    model_name,
    train_patch,
    val_patch,
    wave_cols,
    class_ids,
):
    model = make_classifier(model_name)

    X_train = train_patch[wave_cols].values.astype(float)
    y_train = train_patch["class_id"].astype(int).values

    X_val = val_patch[wave_cols].values.astype(float)

    model.fit(X_train, y_train)

    patch_proba_raw = model.predict_proba(X_val)

    model_classes = model.named_steps["model"].classes_.astype(int)

    aligned_proba = np.zeros(
        (len(val_patch), len(class_ids)),
        dtype=float,
    )

    for j, cls in enumerate(model_classes):
        class_position = class_ids.index(int(cls))
        aligned_proba[:, class_position] = patch_proba_raw[:, j]

    sample_pred = aggregate_patch_probabilities_to_sample(
        patch_df=val_patch,
        patch_probabilities=aligned_proba,
        class_ids=class_ids,
    )

    metrics = calculate_classification_metrics(
        sample_pred=sample_pred,
        class_ids=class_ids,
    )

    metrics["model"] = model_name
    metrics["scenario"] = "fixed_validation"

    return model, sample_pred, metrics


# ============================================================
# 8. Load data
# ============================================================

required_input_files = [
    A_PATCH_CSV,
    B_PATCH_CSV,
    B_QUALITY_XLSX,
]

missing_input_files = [
    path for path in required_input_files if not path.is_file()
]

if missing_input_files:
    missing_text = "\n".join(str(path) for path in missing_input_files)
    raise FileNotFoundError(
        "The following required input files were not found:\n"
        f"{missing_text}"
    )

print("Project root:")
print(PROJECT_DIR)

print("\n2024 flour patch CSV:")
print(A_PATCH_CSV)

print("\n2025 flour patch CSV:")
print(B_PATCH_CSV)

print("\n2025 quality Excel:")
print(B_QUALITY_XLSX)

a_patch, a_wave_cols, a_wavelengths = prepare_patch_table(
    patch_csv=A_PATCH_CSV,
    source_name="A_original",
)

b_patch, b_wave_cols, b_wavelengths = prepare_patch_table(
    patch_csv=B_PATCH_CSV,
    source_name="B_second",
)

if len(a_wave_cols) != len(b_wave_cols):
    raise ValueError(
        f"A and B have different numbers of wavelength columns: "
        f"A={len(a_wave_cols)}, B={len(b_wave_cols)}"
    )

a_waves = np.asarray(a_wavelengths, dtype=float)
b_waves = np.asarray(b_wavelengths, dtype=float)

if not np.allclose(a_waves, b_waves, atol=0.01):
    raise ValueError(
        "A and B wavelength lists are not aligned.\n"
        "Please align/interpolate the spectra before classification."
    )

wavelengths = a_waves
wave_cols = a_wave_cols

b_rename_wave = dict(zip(b_wave_cols, a_wave_cols))
b_patch = b_patch.rename(columns=b_rename_wave)

patch_all = pd.concat(
    [a_patch, b_patch],
    axis=0,
    ignore_index=True,
)

quality_all = load_quality_all_from_original_inputs(
    a_patch_csv=A_PATCH_CSV,
    b_quality_xlsx=B_QUALITY_XLSX,
)

train_ids_protein, val_ids_protein = create_fixed_split_repeat0(
    quality_all=quality_all,
)

print("\nPatch table:")
print(patch_all.shape)
print(patch_all["source"].value_counts())

print("\nQuality table:")
print(quality_all.shape)
print(quality_all["source"].value_counts())

print("\nDetected wavelengths:", len(wavelengths))
print("First 5:", wavelengths[:5])
print("Last 5:", wavelengths[-5:])


# ============================================================
# 9. Run one target
# ============================================================

def run_one_target_individual_classification(target_col):
    target_safe = safe_name(target_col)

    target_out = OUT_ROOT / target_safe
    target_out.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print(f"Target indicator: {target_col}")
    print("Individual KMeans3 classification")
    print("=" * 100)

    class_ids = list(range(K_CLASSES))

    label_df, centers_df, class_count_df, scaler_y, kmeans_y, raw_to_ordered = create_kmeans3_labels_for_target(
        quality_all=quality_all,
        train_ids=train_ids_protein,
        val_ids=val_ids_protein,
        target_col=target_col,
    )

    label_df.to_csv(
        target_out / "sample_level_KMeans3_labels_same_split_as_protein.csv",
        index=False,
    )

    centers_df.to_csv(
        target_out / "KMeans3_class_centers_training_only.csv",
        index=False,
    )

    class_count_df.to_csv(
        target_out / "KMeans3_class_counts_training_validation.csv",
        index=False,
    )

    print("\nKMeans3 class centers:")
    display(centers_df)

    print("\nClass counts:")
    display(class_count_df)

    patch_labeled = prepare_patch_for_classification(
        patch_all=patch_all,
        label_df=label_df,
    )

    train_patch = patch_labeled[
        patch_labeled["global_sample_id"].astype(str).isin(train_ids_protein)
    ].copy()

    val_patch = patch_labeled[
        patch_labeled["global_sample_id"].astype(str).isin(val_ids_protein)
    ].copy()

    if DROP_MISSING_TARGET_IN_SAME_SPLIT:
        train_patch = train_patch.dropna(subset=["class_id"]).copy()
        val_patch = val_patch.dropna(subset=["class_id"]).copy()

    print("\nTraining samples:", train_patch["global_sample_id"].nunique())
    print("Validation samples:", val_patch["global_sample_id"].nunique())
    print("Training patches:", train_patch.shape[0])
    print("Validation patches:", val_patch.shape[0])

    print("\nTraining class distribution:")
    print(
        train_patch
        .drop_duplicates("global_sample_id")["class_name"]
        .value_counts()
        .sort_index()
    )

    print("\nValidation class distribution:")
    print(
        val_patch
        .drop_duplicates("global_sample_id")["class_name"]
        .value_counts()
        .sort_index()
    )

    train_patch.drop_duplicates("global_sample_id")[
        ["global_sample_id", "source", "sample_id", "treatment", "class_id", "class_name", "class_level"]
    ].to_csv(
        target_out / "training_samples_KMeans3_labels_same_split_as_protein.csv",
        index=False,
    )

    val_patch.drop_duplicates("global_sample_id")[
        ["global_sample_id", "source", "sample_id", "treatment", "class_id", "class_name", "class_level"]
    ].to_csv(
        target_out / "validation_samples_KMeans3_labels_same_split_as_protein.csv",
        index=False,
    )

    internal_metrics = []
    oof_prediction_tables = {}

    for model_name in MODEL_NAMES_TO_RUN:
        print(f"\nInternal GroupKFold model: {model_name}")

        try:
            m, oof_pred = groupkfold_oof_classification(
                patch_train=train_patch,
                wave_cols=wave_cols,
                model_name=model_name,
                class_ids=class_ids,
                n_splits=5,
            )

            m["target"] = target_col
            internal_metrics.append(m)

            oof_prediction_tables[model_name] = oof_pred

            print(
                f"{model_name}: "
                f"Accuracy={m['accuracy']:.3f}, "
                f"Balanced accuracy={m['balanced_accuracy']:.3f}, "
                f"Macro-F1={m['macro_f1']:.3f}, "
                f"Weighted-F1={m['weighted_f1']:.3f}"
            )

        except Exception as e:
            print(f"Skipped {model_name} due to error: {e}")

    internal_summary = pd.DataFrame(internal_metrics)

    if internal_summary.empty:
        print(f"No valid internal models for target {target_col}.")
        return None

    internal_summary = internal_summary.sort_values(
        ["macro_f1", "balanced_accuracy", "accuracy"],
        ascending=False,
    ).reset_index(drop=True)

    internal_summary.to_csv(
        target_out / "internal_GroupKFold_summary.csv",
        index=False,
    )

    best_model_name = internal_summary.iloc[0]["model"]

    print("\nBest internal model:")
    print(best_model_name)
    display(internal_summary)

    for model_name, pred_df in oof_prediction_tables.items():
        pred_df.to_csv(
            target_out / f"OOF_predictions_{model_name}.csv",
            index=False,
        )

        report_df = classification_report_df(
            sample_pred=pred_df,
            class_ids=class_ids,
        )

        report_df.to_csv(
            target_out / f"OOF_classification_report_{model_name}.csv",
            index=False,
        )

    best_oof_pred = oof_prediction_tables[best_model_name].copy()

    best_oof_report = classification_report_df(
        sample_pred=best_oof_pred,
        class_ids=class_ids,
    )

    best_oof_report.to_csv(
        target_out / "best_model_OOF_classification_report.csv",
        index=False,
    )

    cm_oof = plot_confusion_matrix(
        sample_pred=best_oof_pred,
        class_ids=class_ids,
        output_path=target_out / "best_model_OOF_confusion_matrix.png",
    )

    pd.DataFrame(
        cm_oof,
        index=[CLASS_ID_TO_NAME[c] for c in class_ids],
        columns=[CLASS_ID_TO_NAME[c] for c in class_ids],
    ).to_csv(
        target_out / "best_model_OOF_confusion_matrix.csv",
    )

    final_model, validation_pred, validation_metrics = fit_final_and_validate_classifier(
        model_name=best_model_name,
        train_patch=train_patch,
        val_patch=val_patch,
        wave_cols=wave_cols,
        class_ids=class_ids,
    )

    validation_metrics["target"] = target_col
    validation_metrics["scenario"] = "same_fixed_split_as_protein"
    validation_metrics["n_train_samples"] = train_patch["global_sample_id"].nunique()
    validation_metrics["n_validation_samples"] = validation_pred["global_sample_id"].nunique()

    validation_metrics_df = pd.DataFrame([validation_metrics])

    validation_metrics_df.to_csv(
        target_out / "fixed_validation_metrics_same_split_as_protein.csv",
        index=False,
    )

    validation_pred.to_csv(
        target_out / "fixed_validation_predictions_same_split_as_protein.csv",
        index=False,
    )

    validation_report_df = classification_report_df(
        sample_pred=validation_pred,
        class_ids=class_ids,
    )

    validation_report_df.to_csv(
        target_out / "fixed_validation_classification_report_same_split_as_protein.csv",
        index=False,
    )

    cm_val = plot_confusion_matrix(
        sample_pred=validation_pred,
        class_ids=class_ids,
        output_path=target_out / "fixed_validation_confusion_matrix_same_split_as_protein.png",
    )

    pd.DataFrame(
        cm_val,
        index=[CLASS_ID_TO_NAME[c] for c in class_ids],
        columns=[CLASS_ID_TO_NAME[c] for c in class_ids],
    ).to_csv(
        target_out / "fixed_validation_confusion_matrix_same_split_as_protein.csv",
    )

    joblib.dump(
        {
            "target": target_col,
            "k_classes": K_CLASSES,
            "class_id_to_name": CLASS_ID_TO_NAME,
            "class_id_to_level": CLASS_ID_TO_LEVEL,
            "label_scaler": scaler_y,
            "label_kmeans": kmeans_y,
            "raw_to_ordered": raw_to_ordered,
            "model_name": best_model_name,
            "model": final_model,
            "wavelengths": wavelengths,
            "wave_cols": wave_cols,
            "internal_summary": internal_summary,
            "train_ids_same_as_protein": train_ids_protein,
            "validation_ids_same_as_protein": val_ids_protein,
            "class_centers": centers_df,
        },
        target_out / f"final_KMeans3_classifier_{best_model_name}_same_split_as_protein.joblib",
    )

    print("\nFixed validation performance:")
    display(validation_metrics_df)

    print("\nFixed validation classification report:")
    display(validation_report_df)

    result = {
        "target": target_col,
        "best_model": best_model_name,

        "internal_accuracy": internal_summary.iloc[0]["accuracy"],
        "internal_balanced_accuracy": internal_summary.iloc[0]["balanced_accuracy"],
        "internal_macro_f1": internal_summary.iloc[0]["macro_f1"],
        "internal_weighted_f1": internal_summary.iloc[0]["weighted_f1"],

        "validation_accuracy": validation_metrics["accuracy"],
        "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
        "validation_macro_f1": validation_metrics["macro_f1"],
        "validation_weighted_f1": validation_metrics["weighted_f1"],

        "n_train_samples": train_patch["global_sample_id"].nunique(),
        "n_validation_samples": validation_pred["global_sample_id"].nunique(),

        "class_I_center": centers_df.loc[centers_df["class_id"] == 0, "center_value_original_scale"].values[0],
        "class_II_center": centers_df.loc[centers_df["class_id"] == 1, "center_value_original_scale"].values[0],
        "class_III_center": centers_df.loc[centers_df["class_id"] == 2, "center_value_original_scale"].values[0],

        "target_output_folder": str(target_out),
    }

    return result


# ============================================================
# 10. Run all 7 targets
# ============================================================

all_target_results = []

for target_col in TARGET_COLS:

    try:
        result = run_one_target_individual_classification(target_col)

        if result is not None:
            all_target_results.append(result)

    except Exception as e:
        print("\n" + "!" * 100)
        print(f"Target failed: {target_col}")
        print("Error:", e)
        print("!" * 100)


# ============================================================
# 11. Overall summary
# ============================================================

overall_summary = pd.DataFrame(all_target_results)

overall_summary_path = OUT_ROOT / "seven_indicator_individual_KMeans3_classification_summary_same_split_as_protein.csv"

overall_summary.to_csv(
    overall_summary_path,
    index=False,
)

print("\n" + "=" * 100)
print("Overall summary across 7 individual indicators")
print("Individual KMeans3 classification")
print("Same fixed split as protein regression")
print("=" * 100)

display(overall_summary)

print("\nSaved overall summary:")
print(overall_summary_path)

print("\nAll outputs saved to:")
print(OUT_ROOT)
